In [1]:
import os
import sys

REPO = "Arabic-sarcasm"

if not os.path.exists(f"/content/{REPO}"):
    !git clone https://github.com/cad0m/{REPO}.git /content/{REPO}
else:
    !git -C /content/{REPO} pull

%cd /content/{REPO}

if f"/content/{REPO}" not in sys.path:
    sys.path.insert(0, f"/content/{REPO}")

!pip install -q uv
!uv pip install --system emoji tashaphyne gensim

Cloning into '/content/Arabic-sarcasm'...
remote: Enumerating objects: 176, done.
remote: Counting objects: 100% (176/176), done.
remote: Compressing objects: 100% (124/124), done.
remote: Total 176 (delta 80), reused 119 (delta 38), pack-reused 0 (from 0)
Receiving objects: 100% (176/176), 1.38 MiB | 4.51 MiB/s, done.
Resolving deltas: 100% (80/80), done.
/content/Arabic-sarcasm
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 96.8 MB/s eta 0:00:00
Using Python 3.13.16 environment at: /usr
Resolved 9 packages in 288ms
Prepared 4 packages in 323ms
Installed 4 packages in 6ms
 + emoji==2.16.0
 + gensim==4.4.0
 + pyarabic==0.6.15
 + tashaphyne==0.3.6


In [2]:
!wget -q -O aravec_utils.py https://raw.githubusercontent.com/bakrianoo/aravec/master/utilities.py
!wget -q -nc -P .. https://bakrianoo.ewr1.vultrobjects.com/aravec/full_uni_sg_100_twitter.zip
!unzip -q -n ../full_uni_sg_100_twitter.zip -d ../aravec

In [21]:
import json
import math
import os
import re
import time
from collections import Counter

import pandas as pd
import torch
from gensim.models import Word2Vec
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence, pad_sequence
from torch.utils.data import DataLoader, TensorDataset

from aravec_utils import clean_str
from src.data import get_data
from src.evaluate import RESULTS, log_result, metrics, save_predictions, show
from src.preprocess import build_pipeline
from src.train import best_threshold, count_params, oversample, set_seed

STEPS = ["html", "urls", "mentions", "diacritics", "kashida", "letters", "elongation"]
PREP = "+".join(STEPS)
SEED = 42

PAD, UNK = 0, 1
MIN_FREQ, MAX_LEN = 2, 64
BATCH_SIZE, LR, MAX_EPOCHS, PATIENCE = 64, 1e-3, 20, 3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs("../models", exist_ok=True)

In [22]:
train, val, test, report = get_data()

prep = build_pipeline(STEPS)
TOKEN = re.compile(r"<\w+>|\w+|[^\w\s]")


def tokenize(tweet):
    return TOKEN.findall(prep(tweet))


train["tokens"] = train["tweet"].apply(tokenize)
val["tokens"] = val["tweet"].apply(tokenize)
test["tokens"] = test["tweet"].apply(tokenize)

In [23]:
n_sarcastic = train["sarcasm"].sum()
n_normal = len(train) - n_sarcastic
WEIGHTS = torch.tensor([len(train) / (2 * n_normal), len(train) / (2 * n_sarcastic)], dtype=torch.float)
WEIGHTS

tensor([0.6040, 2.9044])

In [24]:
freq = Counter()
for tokens in train["tokens"]:
    freq.update(tokens)

vocab = {"<pad>": PAD, "<unk>": UNK}
for word, count in freq.items():
    if count >= MIN_FREQ:
        vocab[word] = len(vocab)

print("vocabulary size:", len(vocab))
print("99th percentile tweet length:", train["tokens"].str.len().quantile(0.99))

vocabulary size: 14777
99th percentile tweet length: 42.0


In [25]:
def oov_rate(frame):
    total, unknown = 0, 0
    for tokens in frame["tokens"]:
        for token in tokens:
            total += 1
            if token not in vocab:
                unknown += 1
    return unknown / total


print(f"OOV val : {oov_rate(val):.2%}")
print(f"OOV test: {oov_rate(test):.2%}")

OOV val : 19.88%
OOV test: 27.32%


In [26]:
def encode(tokens_list):
    rows = []
    for tokens in tokens_list:
        ids = [vocab.get(token, UNK) for token in tokens[:MAX_LEN]]
        if len(ids) == 0:
            ids = [UNK]
        rows.append(torch.tensor(ids))
    return pad_sequence(rows, batch_first=True, padding_value=PAD)


def make_loader(frame, shuffle=False):
    x = encode(frame["tokens"])
    y = torch.tensor(frame["sarcasm"].values)
    return DataLoader(TensorDataset(x, y), batch_size=BATCH_SIZE, shuffle=shuffle)


train_loader = make_loader(train, shuffle=True)
val_loader = make_loader(val)
test_loader = make_loader(test)
y_val = val["sarcasm"].values

In [27]:
wv = Word2Vec.load(next(Path("../aravec").rglob("*.mdl")).as_posix()).wv
DIM = wv.vector_size

In [28]:
set_seed(SEED)
aravec_matrix = torch.randn(len(vocab), DIM)
aravec_matrix[PAD] = 0

found = 0
for word, index in vocab.items():
    key = clean_str(word)
    if key in wv.key_to_index:
        aravec_matrix[index] = torch.from_numpy(wv[key])
        found += 1

print(f"AraVec knows {found / len(vocab):.1%} of our vocabulary")

AraVec knows 94.2% of our vocabulary


In [29]:
def make_embedding(mode):
    if mode == "random":
        return nn.Embedding(len(vocab), DIM, padding_idx=PAD)
    freeze = (mode == "aravec_frozen")
    return nn.Embedding.from_pretrained(aravec_matrix.clone(), freeze=freeze, padding_idx=PAD)

In [30]:
class BiLSTM(nn.Module):
    def __init__(self, embedding, hidden=128):
        super().__init__()
        self.embedding = embedding
        self.lstm = nn.LSTM(embedding.embedding_dim, hidden, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(2 * hidden, 2)

    def forward(self, x):
        mask = x != PAD
        lengths = mask.sum(dim=1).cpu()

        vectors = self.dropout(self.embedding(x))
        packed = pack_padded_sequence(vectors, lengths, batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = pad_packed_sequence(out, batch_first=True, total_length=x.size(1))

        out = out.masked_fill(~mask.unsqueeze(2), -1e9)
        pooled = out.max(dim=1).values
        return self.classifier(self.dropout(pooled))

In [31]:
class BiLSTMAttention(nn.Module):
    def __init__(self, embedding, hidden=128):
        super().__init__()
        self.embedding = embedding
        self.lstm = nn.LSTM(embedding.embedding_dim, hidden, batch_first=True, bidirectional=True)
        self.W = nn.Linear(2 * hidden, 2 * hidden)
        self.v = nn.Linear(2 * hidden, 1, bias=False)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(2 * hidden, 2)

    def forward(self, x):
        mask = x != PAD
        lengths = mask.sum(dim=1).cpu()

        vectors = self.dropout(self.embedding(x))
        packed = pack_padded_sequence(vectors, lengths, batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = pad_packed_sequence(out, batch_first=True, total_length=x.size(1))

        scores = self.v(torch.tanh(self.W(out))).squeeze(2)
        scores = scores.masked_fill(~mask, -1e9)
        alpha = torch.softmax(scores, dim=1)
        self.last_attention = alpha.detach()

        context = (alpha.unsqueeze(2) * out).sum(dim=1)
        return self.classifier(self.dropout(context))

In [32]:
def sinusoidal_positions(length, d_model):
    position = torch.arange(length).unsqueeze(1)
    step = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
    table = torch.zeros(length, d_model)
    table[:, 0::2] = torch.sin(position * step)
    table[:, 1::2] = torch.cos(position * step)
    return table

In [33]:
class MiniTransformer(nn.Module):
    def __init__(self, embedding, use_positions=True, d_model=128):
        super().__init__()
        self.embedding = embedding
        self.project = nn.Linear(embedding.embedding_dim, d_model)
        self.use_positions = use_positions
        self.register_buffer("positions", sinusoidal_positions(MAX_LEN, d_model))
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=4, dim_feedforward=256,
                                           dropout=0.1, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=2, enable_nested_tensor=False)
        self.classifier = nn.Linear(d_model, 2)

    def forward(self, x):
        pad = x == PAD
        h = self.project(self.embedding(x))
        if self.use_positions:
            h = h + self.positions[: x.size(1)]
        h = self.encoder(h, src_key_padding_mask=pad)

        keep = (~pad).unsqueeze(2)
        pooled = (h * keep).sum(dim=1) / keep.sum(dim=1)
        return self.classifier(pooled)

In [34]:
def build(kind, embedding):
    emb = make_embedding(embedding)
    if kind == "c1":
        return BiLSTM(emb)
    if kind == "c2":
        return BiLSTMAttention(emb)
    if kind == "c3":
        return MiniTransformer(emb)
    if kind == "c3_nopos":
        return MiniTransformer(emb, use_positions=False)

In [35]:
def predict(model, loader):
    model.eval()
    probas = []
    with torch.no_grad():
        for x, _ in loader:
            logits = model(x.to(DEVICE))
            probas.append(torch.softmax(logits, dim=1)[:, 1].cpu())
    return torch.cat(probas).numpy()

In [36]:
def train_model(model, name, loader, weights):
    model.to(DEVICE)
    loss_fn = nn.CrossEntropyLoss(weight=weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    best_f1, bad_epochs = 0, 0
    start = time.time()

    for epoch in range(MAX_EPOCHS):
        model.train()
        for x, y in loader:
            loss = loss_fn(model(x.to(DEVICE)), y.to(DEVICE))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        proba = predict(model, val_loader)
        f1 = metrics(y_val, proba, best_threshold(y_val, proba))["f1"]
        print(f"epoch {epoch + 1}  val f1 = {f1:.4f}")

        if f1 > best_f1:
            best_f1, bad_epochs = f1, 0
            torch.save(model.state_dict(), f"../models/{name}.pt")
        else:
            bad_epochs += 1
            if bad_epochs == PATIENCE:
                break

    model.load_state_dict(torch.load(f"../models/{name}.pt"))
    return time.time() - start

In [37]:
def log_run(name, model, seconds):
    proba = predict(model, val_loader)
    threshold = best_threshold(y_val, proba)
    scores = metrics(y_val, proba, threshold)
    log_result(model=name, preprocessing=PREP, seed=SEED, threshold=threshold,
               train_time=seconds, trained_params=count_params(model),
               val_f1=scores["f1"], val_pr_auc=scores["pr_auc"], val_macro_f1=scores["macro_f1"],
               val_precision=scores["precision"], val_recall=scores["recall"],
               val_f1_at_05=metrics(y_val, proba, 0.5)["f1"])
    show(name, scores, threshold, seconds)

In [38]:
def run(kind, embedding, strategy="weighted"):
    set_seed(SEED)
    name = f"{kind}_{embedding}_{strategy}"
    model = build(kind, embedding)

    loader = train_loader
    weights = None
    if strategy == "weighted":
        weights = WEIGHTS.to(DEVICE)
    if strategy == "oversample":
        loader = make_loader(oversample(train, SEED), shuffle=True)

    seconds = train_model(model, name, loader, weights)
    log_run(name, model, seconds)

In [39]:
run("c1", "random")

epoch 1  val f1 = 0.4075
epoch 2  val f1 = 0.4526
epoch 3  val f1 = 0.4676
epoch 4  val f1 = 0.4712
epoch 5  val f1 = 0.4758
epoch 6  val f1 = 0.4845
epoch 7  val f1 = 0.4797
epoch 8  val f1 = 0.4797
epoch 9  val f1 = 0.4811
c1_random_weighted           f1=0.4845  pr_auc=0.4529  threshold=0.67  180s


In [40]:
run("c1", "aravec_frozen")

epoch 1  val f1 = 0.5098
epoch 2  val f1 = 0.5104
epoch 3  val f1 = 0.5277
epoch 4  val f1 = 0.5290
epoch 5  val f1 = 0.5263
epoch 6  val f1 = 0.5298
epoch 7  val f1 = 0.5086
epoch 8  val f1 = 0.5175
epoch 9  val f1 = 0.5010
c1_aravec_frozen_weighted    f1=0.5298  pr_auc=0.5138  threshold=0.77  164s


In [41]:
run("c1", "aravec_tuned")

epoch 1  val f1 = 0.5305
epoch 2  val f1 = 0.5221
epoch 3  val f1 = 0.5367
epoch 4  val f1 = 0.5451
epoch 5  val f1 = 0.5216
epoch 6  val f1 = 0.5083
epoch 7  val f1 = 0.4974
c1_aravec_tuned_weighted     f1=0.5451  pr_auc=0.5233  threshold=0.70  127s


In [42]:
run("c2", "random")

epoch 1  val f1 = 0.4271
epoch 2  val f1 = 0.4349
epoch 3  val f1 = 0.4764
epoch 4  val f1 = 0.4715
epoch 5  val f1 = 0.4742
epoch 6  val f1 = 0.4772
epoch 7  val f1 = 0.4836
epoch 8  val f1 = 0.4799
epoch 9  val f1 = 0.4649
epoch 10  val f1 = 0.4664
c2_random_weighted           f1=0.4836  pr_auc=0.4466  threshold=0.57  221s


In [43]:
run("c2", "aravec_frozen")

epoch 1  val f1 = 0.5000
epoch 2  val f1 = 0.5131
epoch 3  val f1 = 0.5186
epoch 4  val f1 = 0.5236
epoch 5  val f1 = 0.5092
epoch 6  val f1 = 0.4877
epoch 7  val f1 = 0.5062
c2_aravec_frozen_weighted    f1=0.5236  pr_auc=0.5126  threshold=0.60  145s


In [44]:
run("c2", "aravec_tuned")

epoch 1  val f1 = 0.5070
epoch 2  val f1 = 0.5171
epoch 3  val f1 = 0.4977
epoch 4  val f1 = 0.5097
epoch 5  val f1 = 0.5150
c2_aravec_tuned_weighted     f1=0.5171  pr_auc=0.5239  threshold=0.52  115s


In [45]:
run("c3", "random")

epoch 1  val f1 = 0.4506
epoch 2  val f1 = 0.4646
epoch 3  val f1 = 0.4971
epoch 4  val f1 = 0.4823
epoch 5  val f1 = 0.4735
epoch 6  val f1 = 0.4678
c3_random_weighted           f1=0.4971  pr_auc=0.4612  threshold=0.50  237s


In [46]:
run("c3", "aravec_frozen")

epoch 1  val f1 = 0.5181
epoch 2  val f1 = 0.5223
epoch 3  val f1 = 0.5271
epoch 4  val f1 = 0.5642
epoch 5  val f1 = 0.5531
epoch 6  val f1 = 0.5296
epoch 7  val f1 = 0.5245
c3_aravec_frozen_weighted    f1=0.5642  pr_auc=0.5654  threshold=0.52  269s


In [47]:
run("c3", "aravec_tuned")

epoch 1  val f1 = 0.5179
epoch 2  val f1 = 0.5462
epoch 3  val f1 = 0.5280
epoch 4  val f1 = 0.4973
epoch 5  val f1 = 0.4494
c3_aravec_tuned_weighted     f1=0.5462  pr_auc=0.5572  threshold=0.72  192s


In [48]:
res = pd.read_csv(RESULTS)
res[res["model"].str.match(r"c\d_")][
    ["model", "val_f1", "val_pr_auc", "val_f1_at_05", "trained_params", "train_time"]
].sort_values("val_f1", ascending=False)

,model,val_f1,val_pr_auc,val_f1_at_05,trained_params,train_time
114,c3_aravec_frozen_weighted,0.564211,0.565393,0.555102,278146,268.723615
115,c3_aravec_tuned_weighted,0.546185,0.557231,0.475196,1755846,192.241253
109,c1_aravec_tuned_weighted,0.545090,0.523277,0.511628,1713734,127.073177
108,c1_aravec_frozen_weighted,0.529774,0.513837,0.468514,236034,164.048357
111,c2_aravec_frozen_weighted,0.523649,0.512580,0.493036,302082,145.342244
112,c2_aravec_tuned_weighted,0.517123,0.523853,0.510924,1779782,114.776273
113,c3_random_weighted,0.497065,0.461197,0.494118,1755846,236.749571
107,c1_random_weighted,0.484517,0.452874,0.473445,1713734,180.296134
110,c2_random_weighted,0.483559,0.446601,0.468468,1779782,221.159149


In [49]:
run("c3_nopos", "aravec_tuned")

epoch 1  val f1 = 0.5200
epoch 2  val f1 = 0.5269
epoch 3  val f1 = 0.4847
epoch 4  val f1 = 0.4812
epoch 5  val f1 = 0.4776
c3_nopos_aravec_tuned_weighted f1=0.5269  pr_auc=0.4997  threshold=0.73  191s


In [50]:
run("c2", "aravec_tuned", "oversample")

epoch 1  val f1 = 0.5174
epoch 2  val f1 = 0.4916
epoch 3  val f1 = 0.4884
epoch 4  val f1 = 0.4869
c2_aravec_tuned_oversample   f1=0.5174  pr_auc=0.5066  threshold=0.66  148s


In [51]:
run("c2", "aravec_tuned", "standard")

epoch 1  val f1 = 0.4983
epoch 2  val f1 = 0.5261
epoch 3  val f1 = 0.5152
epoch 4  val f1 = 0.4974
epoch 5  val f1 = 0.5098
c2_aravec_tuned_standard     f1=0.5261  pr_auc=0.5326  threshold=0.40  106s


In [52]:
with open("../models/vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False)

In [53]:
def final_test(kind, embedding, strategy="weighted"):
    name = f"{kind}_{embedding}_{strategy}"
    model = build(kind, embedding).to(DEVICE)
    model.load_state_dict(torch.load(f"../models/{name}.pt"))

    threshold = best_threshold(y_val, predict(model, val_loader))
    test_proba = predict(model, test_loader)
    save_predictions(name, SEED, test["id"], test["sarcasm"], test_proba, threshold)
    print(name, metrics(test["sarcasm"], test_proba, threshold))


final_test("c2", "aravec_tuned", "weighted")

c2_aravec_tuned_weighted {'f1': 0.5590361445783133, 'pr_auc': np.float64(0.5545407143401091), 'macro_f1': 0.6629575627350166, 'precision': 0.4625199362041467, 'recall': 0.7064555420219245}
